In [2]:
from keras.utils import to_categorical, load_img, img_to_array
from keras.models import Sequential
from keras.layers import Dense, Conv2D, Dropout, Flatten, MaxPooling2D
import os
import pandas as pd
import numpy as np

In [3]:
TRAIN_DIR = 'images/train'
TEST_DIR = 'images/test'

In [4]:
def createdataframe(dir):
    image_paths = []
    labels = []
    for label in os.listdir(dir):
        for imagename in os.listdir(os.path.join(dir,label)):
            image_paths.append(os.path.join(dir,label,imagename))
            labels.append(label)
        print(label, "completed")
    return image_paths, labels

In [5]:
train = pd.DataFrame()
train['image'], train['label'] = createdataframe(TRAIN_DIR)

angry completed
disgust completed
fear completed
happy completed
neutral completed
sad completed
surprise completed


In [6]:
print(train)

                                image     label
0            images/train\angry\0.jpg     angry
1            images/train\angry\1.jpg     angry
2           images/train\angry\10.jpg     angry
3        images/train\angry\10002.jpg     angry
4        images/train\angry\10016.jpg     angry
...                               ...       ...
28816  images/train\surprise\9969.jpg  surprise
28817  images/train\surprise\9985.jpg  surprise
28818  images/train\surprise\9990.jpg  surprise
28819  images/train\surprise\9992.jpg  surprise
28820  images/train\surprise\9996.jpg  surprise

[28821 rows x 2 columns]


In [7]:
test = pd.DataFrame()
test['image'], test['label'] = createdataframe(TEST_DIR)

angry completed
disgust completed
fear completed
happy completed
neutral completed
sad completed
surprise completed


In [8]:
print(test)

                              image     label
0       images/test\angry\10052.jpg     angry
1       images/test\angry\10065.jpg     angry
2       images/test\angry\10079.jpg     angry
3       images/test\angry\10095.jpg     angry
4       images/test\angry\10121.jpg     angry
...                             ...       ...
7061  images/test\surprise\9806.jpg  surprise
7062  images/test\surprise\9830.jpg  surprise
7063  images/test\surprise\9853.jpg  surprise
7064  images/test\surprise\9878.jpg  surprise
7065   images/test\surprise\993.jpg  surprise

[7066 rows x 2 columns]


In [9]:
from tqdm.notebook import tqdm

In [10]:
def extract_features(images):
    features = []
    for image in tqdm(images):
        img = load_img(image,color_mode='grayscale')
        img = np.array(img)
        features.append(img)
    features = np.array(features)
    features = features.reshape(len(features),48,48,1)
    return features

In [11]:
train_features = extract_features(train['image'])

  0%|          | 0/28821 [00:00<?, ?it/s]

In [12]:
test_features = extract_features(test['image'])

  0%|          | 0/7066 [00:00<?, ?it/s]

In [13]:
x_train = train_features/255.0
x_test = test_features/255.0

In [14]:
from sklearn.preprocessing import LabelEncoder

In [15]:
le = LabelEncoder()
le.fit(train['label'])

LabelEncoder()

In [16]:
y_train = le.transform(train['label'])
y_test = le.transform(test['label'])

In [17]:
y_train = to_categorical(y_train, num_classes = 7)
y_test = to_categorical(y_test, num_classes = 7)

In [25]:
model = Sequential()

# convolutional layers
model.add(Conv2D(128, kernel_size=(3,3), activation='relu', input_shape=(48,48,1)))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(256, kernel_size=(3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(512, kernel_size=(3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(512, kernel_size=(3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Flatten())

# fully connected layers
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.4))
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.3))

# output layer
model.add(Dense(7, activation='softmax'))

In [26]:
model.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'] )

In [27]:
#checking integrated graphics as gpu because I dont have a External GPU

import tensorflow as tf
import numpy as np

print(f"TensorFlow Version: {tf.__version__}")
print(f"NumPy Version:      {np.__version__}")

physical_devices = tf.config.list_physical_devices()
print("\nDetected Physical Devices:")
for device in physical_devices:
    print(f" - {device}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\nDirectML has mapped your Intel Iris Xe Graphics as a GPU device.")
else:
    print("\nDirectML GPU device not found. Running on CPU mode.")

TensorFlow Version: 2.10.0
NumPy Version:      1.23.5

Detected Physical Devices:
 - PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')
 - PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

DirectML has mapped your Intel Iris Xe Graphics as a GPU device.


In [29]:
from keras.callbacks import EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model.fit(x = x_train, y = y_train, batch_size = 120, epochs = 30, validation_data = (x_test, y_test), callbacks = [early_stop])

Epoch 1/30
241/241 [==============================] - 155s 549ms/step - loss: 1.8206 - accuracy: 0.2439 - val_loss: 1.8014 - val_accuracy: 0.2586
Epoch 2/30
241/241 [==============================] - 129s 535ms/step - loss: 1.7728 - accuracy: 0.2622 - val_loss: 1.6762 - val_accuracy: 0.3098
Epoch 3/30
241/241 [==============================] - 139s 576ms/step - loss: 1.6500 - accuracy: 0.3375 - val_loss: 1.4974 - val_accuracy: 0.4189
Epoch 4/30
241/241 [==============================] - 135s 561ms/step - loss: 1.5242 - accuracy: 0.4055 - val_loss: 1.4029 - val_accuracy: 0.4563
Epoch 5/30
241/241 [==============================] - 129s 537ms/step - loss: 1.4472 - accuracy: 0.4391 - val_loss: 1.3252 - val_accuracy: 0.4841
Epoch 6/30
241/241 [==============================] - 129s 535ms/step - loss: 1.3992 - accuracy: 0.4581 - val_loss: 1.2804 - val_accuracy: 0.5144
Epoch 7/30
241/241 [==============================] - 129s 536ms/step - loss: 1.3587 - accuracy: 0.4758 - val_loss: 1.2589 -

In [ ]:
model_json = model.to_json()
with open("emotion_detector_model_0.json",'w') as json_file:
    json_file.write(model_json)
model.save("emotion_detector_model_0.h5")

In [1]:
from keras.models import model_from_json

In [2]:
json_file = open("emotion_detector_model_0.json","r")
model_json = json_file.read()
json_file.close()
model = model_from_json(model_json)
model.load_weights("emotion_detector_model_0.h5")

In [3]:
label = ['angry','disgust','fear','happy','neutral','sad','surprise']

In [4]:
def ef(image): # ef = extract features
    img = load_img(image, color_mode='grayscale')
    feature = np.array(img)
    feature = feature.reshape(1, 48, 48, 1)
    return feature / 255.0

In [6]:
#import matplotlib.pyplot as plt
#%matplotlib inline

In [17]:
image = "images/train/neutral/12.jpg"
img = ef(image)
prediction = model.predict(img)
prediction_label = label[prediction.argmax()]
print("Model prediction is ",prediction_label)
#plt.imshow(img.reshape(48,48),cmap='gray')

1/1 [==============================] - 0s 55ms/step
Model prediction is  neutral


In [27]:
# previous model ("emotion_detector_model_0.json") gave 60% accuracy, now training a new model with changes in training architecture:

In [23]:
# updated architecture:

import tensorflow as tf
from keras.models import Sequential
from keras.layers import (
    Dense, Conv2D, Dropout, MaxPooling2D, 
    BatchNormalization, GlobalAveragePooling2D
)
from keras.regularizers import l2
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.preprocessing.image import ImageDataGenerator

model = Sequential()

# block 1: basic edges & textures (48x48 -> 24x24)
model.add(Conv2D(64, kernel_size=(3, 3), padding='same', input_shape=(48, 48, 1)))
model.add(BatchNormalization())
model.add(tf.keras.layers.Activation('relu'))
model.add(Conv2D(64, kernel_size=(3, 3), padding='same'))
model.add(BatchNormalization())
model.add(tf.keras.layers.Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

# block 2: facial structure (24x24 -> 12x12)
model.add(Conv2D(128, kernel_size=(3, 3), padding='same'))
model.add(BatchNormalization())
model.add(tf.keras.layers.Activation('relu'))
model.add(Conv2D(128, kernel_size=(3, 3), padding='same'))
model.add(BatchNormalization())
model.add(tf.keras.layers.Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.30))

# block 3: expression patterns (12x12 -> 6x6)
model.add(Conv2D(256, kernel_size=(3, 3), padding='same'))
model.add(BatchNormalization())
model.add(tf.keras.layers.Activation('relu'))
model.add(Conv2D(256, kernel_size=(3, 3), padding='same'))
model.add(BatchNormalization())
model.add(tf.keras.layers.Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.40))

# global feature pooling
model.add(GlobalAveragePooling2D())

# fully connected classifier
model.add(Dense(256, kernel_regularizer=l2(0.01)))
model.add(BatchNormalization())
model.add(tf.keras.layers.Activation('relu'))
model.add(Dropout(0.50))

# uutput layer: 7 emotions
model.add(Dense(7, activation='softmax'))

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 2. data augmentation
train_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest'
)

# 3. callbacks
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

callbacks = [lr_scheduler, early_stop]

In [24]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [25]:
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Callbacks setup
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

# fit model with augmented data generator
history = model.fit(
    train_datagen.flow(x_train, y_train, batch_size=120),
    epochs=70,
    validation_data=(x_test, y_test),
    callbacks=[lr_scheduler, early_stop]
)

Epoch 1/70
241/241 [==============================] - 208s 823ms/step - loss: 3.0522 - accuracy: 0.2172 - val_loss: 2.0597 - val_accuracy: 0.2586 - lr: 0.0010
Epoch 2/70
241/241 [==============================] - 186s 773ms/step - loss: 1.9725 - accuracy: 0.2488 - val_loss: 1.9453 - val_accuracy: 0.2645 - lr: 0.0010
Epoch 3/70
241/241 [==============================] - 189s 786ms/step - loss: 1.7660 - accuracy: 0.3093 - val_loss: 1.7701 - val_accuracy: 0.3139 - lr: 0.0010
Epoch 4/70
241/241 [==============================] - 188s 780ms/step - loss: 1.6172 - accuracy: 0.3932 - val_loss: 1.8318 - val_accuracy: 0.3136 - lr: 0.0010
Epoch 5/70
241/241 [==============================] - 188s 778ms/step - loss: 1.4776 - accuracy: 0.4589 - val_loss: 1.8277 - val_accuracy: 0.3607 - lr: 0.0010
Epoch 6/70
241/241 [==============================] - 188s 779ms/step - loss: 1.4010 - accuracy: 0.4889 - val_loss: 1.3083 - val_accuracy: 0.5245 - lr: 0.0010
Epoch 7/70
241/241 [==========================

In [29]:
model_json = model.to_json()
with open("emotion_detector_model_1.json",'w') as json_file:
    json_file.write(model_json)
model.save("emotion_detector_model_1.h5")

In [ ]:
# the new model ("emotion_detector_model_1.h5") got an accuracy of 66.5%